# Обрезка технических 5′ RACE праймеров — овца (`PRJNA900592`)

Вход — обновлённая стадия `trimmed` после cutadapt adapters-only → fastp Q30/u40. `cutadapt` удаляет SMARTer anchor и опубликованные C-region праймеры без дополнительной фильтрации. Стадия `pr_trimmed` заменяется атомарно после проверки всех пар. QC запускается отдельно.


In [ ]:
import os, sys, sysconfig, shutil, subprocess, time
from pathlib import Path
_CONDA_ENV = "/opt/conda/envs/bcr_env"
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [_CONDA_ENV + "/lib/python3.11/site-packages", sysconfig.get_path("purelib")]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
os.makedirs(os.environ["XDG_CONFIG_HOME"], exist_ok=True)


In [ ]:
ADAPTER_MIN_OVERLAP = 10
FIVE_RACE_TECHNICAL_PRIMERS = [
    'CTAATACGACTCACTATAGGGCAAGCAGTGGTATCAACGCAGAGT',
    'TCCCCGCAGCAAGAAGTCAGAGGGTAG',
    'GTTTGAAGAGGAAGACGGATGGCTGAGC',
    'AGGGTGACCGAGGGTGCGGACTT',
]


In [ ]:
def _tool(name):
    p=shutil.which(name)
    if not p: raise FileNotFoundError(name)
    return p

def _run_visible(cmd, stdout_log, stderr_log, outputs=(), heartbeat=30):
    started=time.monotonic(); print('[run]', ' '.join(map(str,cmd)), flush=True)
    with open(stdout_log,'w') as stdout, open(stderr_log,'w') as stderr:
        proc=subprocess.Popen([str(x) for x in cmd],stdout=stdout,stderr=stderr,text=True)
        print(f'PID={proc.pid}',flush=True)
        while proc.poll() is None:
            sizes=' '.join(f'{Path(x).name}={Path(x).stat().st_size/1e6:.1f}MB' for x in outputs if Path(x).exists())
            print(f'PID={proc.pid} elapsed={(time.monotonic()-started)/60:.1f}min {sizes}',flush=True)
            time.sleep(heartbeat)
    if proc.returncode: raise RuntimeError(f'rc={proc.returncode}; see {stderr_log}')

def _promote(staging,final):
    previous=final.parent/f'.{final.name}.previous'
    if previous.exists(): shutil.rmtree(previous)
    if final.exists(): final.rename(previous)
    try: staging.rename(final)
    except Exception:
        if previous.exists() and not final.exists(): previous.rename(final)
        raise
    if previous.exists(): shutil.rmtree(previous)

def run_primer_trim_sheep(volume, dataset='PRJNA900592', force=False):
    vol=Path(volume); src=vol/'results'/dataset/'trimmed'/'fastq'
    final=vol/'results'/dataset/'pr_trimmed'; staging=final.parent/'.pr_trimmed.staging'
    pairs=sorted(p.name.removesuffix('_1.trim.fastq.gz') for p in src.glob('*_1.trim.fastq.gz'))
    if not pairs or any(not (src/f'{bn}_2.trim.fastq.gz').is_file() for bn in pairs): raise RuntimeError(f'Incomplete paired inputs in {src}')
    if staging.exists():
        if not force: raise FileExistsError(staging)
        shutil.rmtree(staging)
    out=staging/'fastq'; logs=staging/'cutadapt_reports'
    out.mkdir(parents=True); logs.mkdir(parents=True)
    for bn in pairs:
        r1=src/f'{bn}_1.trim.fastq.gz'; r2=src/f'{bn}_2.trim.fastq.gz'; o1=out/f'{bn}_1.pr.fastq.gz'; o2=out/f'{bn}_2.pr.fastq.gz'
        cmd=[_tool('cutadapt'),'--compression-level','1','-O',str(ADAPTER_MIN_OVERLAP)]
        for s in FIVE_RACE_TECHNICAL_PRIMERS: cmd += ['-g','^'+s,'-G','^'+s]
        cmd += ['--json',logs/f'{bn}.cutadapt.json','-o',o1,'-p',o2,r1,r2]
        _run_visible(cmd,logs/f'{bn}.stdout.log',logs/f'{bn}.stderr.log',[o1,o2])
    if len(list(out.glob('*.pr.fastq.gz'))) != 2*len(pairs): raise RuntimeError('Primer output completeness validation failed')
    _promote(staging,final)
    print(f'[primer_trim_sheep] DONE and promoted: {final}',flush=True)


## Запуск с атомарной заменой старой стадии


In [ ]:
run_primer_trim_sheep('/data/user/epishkin', 'PRJNA900592', force=True)
